In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from pydantic_ai import Agent
from typing import List, Any

In [3]:
import json

with open("my_chunks_sections.json", "r", encoding="utf-8") as f:
    my_chunks_sections = json.load(f)

len(my_chunks_sections)
my_chunks_sections[0]

{'filename': 'binance-spot-api-docs-master/CHANGELOG_CN.md',
 'section': '## REST API\n\n- `POST /api/v3/userDataStream`\n- `PUT /api/v3/userDataStream`\n- `DELETE /api/v3/userDataStream`'}

In [4]:
from minsearch import Index, VectorSearch
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
import numpy as np

In [5]:
def normalize_binance_chunks(chunks):
    normalized = []

    for i, doc in enumerate(chunks):
        item = dict(doc)

        chunk_text = (
            item.get("chunk")
            or item.get("content")
            or item.get("text")
            or ""
        )

        title = item.get("title", "") or item.get("heading", "") or ""
        filename = (
            item.get("filename", "")
            or item.get("file_path", "")
            or item.get("path", "")
            or item.get("source", "")
            or ""
        )
        section = item.get("section", "") or item.get("header", "") or ""
        description = item.get("description", "") or ""

        normalized.append({
            "id": item.get("id", str(i)),
            "chunk": chunk_text,
            "title": title,
            "filename": filename,
            "section": section,
            "description": description,
            "url": item.get("url", ""),
            "raw": item
        })

    return normalized

In [6]:
binance_docs = []

for i, d in enumerate(my_chunks_sections):
    binance_docs.append({
        "id": str(i),
        "chunk": d.get("chunk", "") if isinstance(d, dict) else "",
        "title": d.get("title", "") if isinstance(d, dict) else "",
        "description": d.get("description", "") if isinstance(d, dict) else "",
        "filename": d.get("filename", "") if isinstance(d, dict) else "",
        "section": d.get("section", "") if isinstance(d, dict) else ""
    })

len(binance_docs)

367

In [7]:
binance_docs[0]

{'id': '0',
 'chunk': '',
 'title': '',
 'description': '',
 'filename': 'binance-spot-api-docs-master/CHANGELOG_CN.md',
 'section': '## REST API\n\n- `POST /api/v3/userDataStream`\n- `PUT /api/v3/userDataStream`\n- `DELETE /api/v3/userDataStream`'}

In [8]:
text_index = Index(
    text_fields=["chunk", "title", "description", "filename", "section"],
    keyword_fields=[]
)

text_index.fit(binance_docs)

In [9]:
query = "How do I sign a Binance Spot API request?"
text_results = text_index.search(query, num_results=5)
text_results

[{'id': '170',
  'chunk': '',
  'title': '',
  'description': '',
  'filename': 'binance-spot-api-docs-master/fix-api.md',
  'section': '## General API Information\n\n* FIX connections require TLS encryption. Please either use native TCP+TLS connection or set up a local proxy such as [stunnel](https://www.stunnel.org/) to handle TLS encryption.\n* APIs have a timeout of 10 seconds when processing a request. If a response from the Matching Engine takes longer than this, the API responds with "Timeout waiting for response from backend server. Send status unknown; execution status unknown." [(-1007 TIMEOUT)](errors.md#-1007-timeout)\n  * This does not always mean that the request failed in the Matching Engine.\n  * If the status of the request has not appeared in [User Data Stream](user-data-stream.md), please perform an API query for its status.\n* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.\

In [10]:
def preview_results(results, max_chars=300):
    for i, r in enumerate(results, 1):
        print(f"\nResult #{i}")
        print("title:      ", r.get("title", ""))
        print("section:    ", r.get("section", ""))
        print("filename:   ", r.get("filename", ""))
        print("chunk:      ", r.get("chunk", "")[:max_chars])
        print("-" * 80)

In [11]:
preview_results(text_results)


Result #1
title:       
section:     ## General API Information

* FIX connections require TLS encryption. Please either use native TCP+TLS connection or set up a local proxy such as [stunnel](https://www.stunnel.org/) to handle TLS encryption.
* APIs have a timeout of 10 seconds when processing a request. If a response from the Matching Engine takes longer than this, the API responds with "Timeout waiting for response from backend server. Send status unknown; execution status unknown." [(-1007 TIMEOUT)](errors.md#-1007-timeout)
  * This does not always mean that the request failed in the Matching Engine.
  * If the status of the request has not appeared in [User Data Stream](user-data-stream.md), please perform an API query for its status.
* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.
* To ensure uninterrupted connectivity, please make sure that your client sends **SNI (Server Name Indica

In [12]:
embedding_model = SentenceTransformer("multi-qa-distilbert-cos-v1")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [13]:
binance_embeddings = []

for d in tqdm(binance_docs):
    v = embedding_model.encode(d["chunk"])
    binance_embeddings.append(v)

binance_embeddings = np.array(binance_embeddings)
binance_embeddings.shape

  0%|          | 0/367 [00:00<?, ?it/s]

(367, 768)

In [14]:
vector_index = VectorSearch()
vector_index.fit(binance_embeddings, binance_docs)

In [15]:
query = "How can I authenticate a private Binance API endpoint?"
q = embedding_model.encode(query)

vector_results = vector_index.search(q, num_results=5)
preview_results(vector_results)


Result #1
title:       
section:     ## 如何正确在本地维护一个order book副本

1. 打开与 `wss://stream.binance.com:9443/ws/bnbbtc@depth` 的 WebSocket 连接。
2. 开始缓存收到的event。请记录您收到的第一个event的 `U`值。
3. 访问 `https://api.binance.com/api/v3/depth?symbol=BNBBTC&limit=5000` 获取深度快照。
4. 如果快照中的 `lastUpdateId` 小于等于步骤 2 中的 `U` 值，请返回步骤 3。
5. 在收到的event中，丢弃快照中 `u` <= `lastUpdateId` 的所有event。现在第一个event的 `lastUpdateId` 应该在 `[U;u]` 范围以内。
6. 将本地order book设置为快照。它的更新ID 为 `lastUpdateId`。
7. 更新所有缓存的event，以及后续的所有event。

要将一个event应用于您的本地order book，请遵循以下更新过程：
1. 判断是否需要处理event：
    * 如果event的最后一次更新ID（`u`）小于本地order book的更新ID，忽略该event。
    * 如果event的首次更新ID（`U`）大于本地order book的更新ID加1，说明你错过了一些events。<br>请丢弃您的本地order book并从头开始重新同步。
    * 通常，下一event的`U`等于上一event的`u + 1`。
1. 对买价（`b`）和卖价（`a`）中的每个价位，设置order book中的新数量：
    * 如果该价位在order book中不存在，则插入该价位及其数量。
    * 如果数量为零，则从order book中删除此价位。
1. 将order book的更新ID设置为已处理event的最后一次更新ID（`u`）。

> [!NOTE]
> 由于从 API 检索的深度快照对价位的数量有限制（每侧最多 5000 个），因此除非它们发生变化，否则您将无法了解初始快照之外的价位数量。<br>
> 因此，在使用这些级别的信息时要小心，因为它们

In [16]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)


def vector_search(query, num_results=5):
    q = embedding_model.encode(query)
    return vector_index.search(q, num_results=num_results)

In [17]:
def hybrid_search(query, num_results=5):
    text_results = text_search(query, num_results=num_results)
    vector_results = vector_search(query, num_results=num_results)

    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result["id"] not in seen_ids:
            seen_ids.add(result["id"])
            combined_results.append(result)

    return combined_results[:num_results]

In [18]:
query = "What does recvWindow mean in Binance API requests?"
hybrid_results = hybrid_search(query, num_results=5)
preview_results(hybrid_results)


Result #1
title:       
section:     ## General API Information

* The following base endpoints are available. Please use whichever works best for your setup:
  * **https://api.binance.com**
  * **https://api-gcp.binance.com**
  * **https://api1.binance.com**
  * **https://api2.binance.com**
  * **https://api3.binance.com**
  * **https://api4.binance.com**
* The last 4 endpoints in the point above (`api1`-`api4`) should give better performance but have less stability.
* Responses are in JSON by default. To receive responses in SBE, refer to the [SBE FAQ](./faqs/sbe_faq.md) page.
* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.
* Some endpoints may return asset and/or symbol names containing non-ASCII characters encoded in UTF-8 even if the request did not contain non-ASCII characters.
* Data is returned in **chronological order**, unless noted otherwise.
  * Without `startTime` or `endTime`, 

In [19]:
test_queries = [
    "How do I sign a Binance Spot API request?",
    "What does recvWindow mean?",
    "Why am I getting timestamp error?",
    "How do I place a market order?",
    "How can I authenticate a private endpoint?",
    "What are the rate limits?",
    "How do I get exchange info?"
]

for query in test_queries:
    print("\n" + "=" * 100)
    print("QUERY:", query)
    results = hybrid_search(query, num_results=3)
    preview_results(results, max_chars=220)


QUERY: How do I sign a Binance Spot API request?

Result #1
title:       
section:     ## General API Information

* FIX connections require TLS encryption. Please either use native TCP+TLS connection or set up a local proxy such as [stunnel](https://www.stunnel.org/) to handle TLS encryption.
* APIs have a timeout of 10 seconds when processing a request. If a response from the Matching Engine takes longer than this, the API responds with "Timeout waiting for response from backend server. Send status unknown; execution status unknown." [(-1007 TIMEOUT)](errors.md#-1007-timeout)
  * This does not always mean that the request failed in the Matching Engine.
  * If the status of the request has not appeared in [User Data Stream](user-data-stream.md), please perform an API query for its status.
* If your request contains a symbol name containing non-ASCII characters, then the response may contain non-ASCII characters encoded in UTF-8.
* To ensure uninterrupted connectivity, please make sur

In [20]:
len(my_chunks_sections)

367

In [21]:
preview_results(text_search("What does recvWindow mean?"))


Result #1
title:       
section:     ## In the API response, there's a field called `workingFloor`. What does that field mean?

This is a term used to determine where the order's last activity occurred (filling, expiring, or being placed as new, etc.).

If the `workingFloor` is `SOR`, this means your order interacted with other eligible order books in the SOR configuration.

If the `workingFloor` is `EXCHANGE`, this means your order interacted on the order book that you sent that order to.
filename:    binance-spot-api-docs-master/faqs/sor_faq.md
chunk:       
--------------------------------------------------------------------------------

Result #2
title:       
section:     ## What does the Price Range Execution Rule do?

This rule ensures that trades may only be executed at prices within and equal to a price range around a reference price.
filename:    binance-spot-api-docs-master/faqs/price_range_execution_rules.md
chunk:       
---------------------------------------------------

In [22]:
preview_results(vector_search("How do I authenticate a private endpoint?"))


Result #1
title:       
section:     ## 如何正确在本地维护一个order book副本

1. 打开与 `wss://stream.binance.com:9443/ws/bnbbtc@depth` 的 WebSocket 连接。
2. 开始缓存收到的event。请记录您收到的第一个event的 `U`值。
3. 访问 `https://api.binance.com/api/v3/depth?symbol=BNBBTC&limit=5000` 获取深度快照。
4. 如果快照中的 `lastUpdateId` 小于等于步骤 2 中的 `U` 值，请返回步骤 3。
5. 在收到的event中，丢弃快照中 `u` <= `lastUpdateId` 的所有event。现在第一个event的 `lastUpdateId` 应该在 `[U;u]` 范围以内。
6. 将本地order book设置为快照。它的更新ID 为 `lastUpdateId`。
7. 更新所有缓存的event，以及后续的所有event。

要将一个event应用于您的本地order book，请遵循以下更新过程：
1. 判断是否需要处理event：
    * 如果event的最后一次更新ID（`u`）小于本地order book的更新ID，忽略该event。
    * 如果event的首次更新ID（`U`）大于本地order book的更新ID加1，说明你错过了一些events。<br>请丢弃您的本地order book并从头开始重新同步。
    * 通常，下一event的`U`等于上一event的`u + 1`。
1. 对买价（`b`）和卖价（`a`）中的每个价位，设置order book中的新数量：
    * 如果该价位在order book中不存在，则插入该价位及其数量。
    * 如果数量为零，则从order book中删除此价位。
1. 将order book的更新ID设置为已处理event的最后一次更新ID（`u`）。

> [!NOTE]
> 由于从 API 检索的深度快照对价位的数量有限制（每侧最多 5000 个），因此除非它们发生变化，否则您将无法了解初始快照之外的价位数量。<br>
> 因此，在使用这些级别的信息时要小心，因为它们

In [23]:
preview_results(hybrid_search("Why am I getting timestamp error?"))


Result #1
title:       
section:     ## Error Codes

* Any endpoint can return an ERROR

Sample Payload below:
```javascript
{
    "code": -1121,
    "msg": "Invalid symbol."
}
```
* Specific error codes and messages are defined in [Errors Codes](./errors.md).
filename:    binance-spot-api-docs-master/rest-api.md
chunk:       
--------------------------------------------------------------------------------

Result #2
title:       
section:     ## Error Codes

* Any endpoint can return an ERROR

Sample Payload below:
```javascript
{
    "code": -1121,
    "msg": "Invalid symbol."
}
```
* Specific error codes and messages are defined in [Errors Codes](./errors.md).
filename:    binance-spot-api-docs-master/testnet/rest-api.md
chunk:       
--------------------------------------------------------------------------------

Result #3
title:       
section:     ## 追踪止盈止损下单示例

假设当前最新价是40,000.

下一个追踪止损(`STOP_LOSS_LIMIT`)的买单, 设置价格为42,000, 追踪价差（`trailingDelta`)为5%.

```bash
# 不设置止损价(stop price)


In [24]:
#Text search works well for exact Binance terms like recvWindow, timestamp, and endpoint names.
#TVector search helps when the question is paraphrased.
#THybrid search is the safest overall.
#TFor this project, I would start with text search and keep hybrid as the stronger production option.

In [25]:
print([name for name in dir() if "search" in name.lower()])

['VectorSearch', 'hybrid_search', 'text_search', 'vector_search']


In [26]:
import inspect

print(inspect.signature(text_search))
print(inspect.signature(vector_search))
print(inspect.signature(hybrid_search))

(query, num_results=5)
(query, num_results=5)
(query, num_results=5)


In [27]:
def hybrid_search(query: str, num_results: int = 5):
    text_results = text_search(query)
    vector_results = vector_search(query, num_results=num_results)

    seen_ids = set()
    combined = []

    for r in text_results + vector_results:
        if isinstance(r, dict):
            rid = r.get("id") or r.get("chunk_id") or str(r)
        else:
            rid = str(r)

        if rid not in seen_ids:
            seen_ids.add(rid)
            combined.append(r)

    return combined[:num_results]

In [28]:
print(hybrid_search("authentication"))

[{'id': '297', 'chunk': '', 'title': '', 'description': '', 'filename': 'binance-spot-api-docs-master/user-data-stream.md', 'section': '## Event Stream Terminated\n\n`eventStreamTerminated` is sent when:\n* [A listen token subscription](https://developers.binance.com/docs/margin_trading/trade-data-stream/Listen-Token-Websocket-API#subscribe-to-user-data-stream-using-listentoken-user_stream) expires due to token expiration.\n* A [logon subscription](https://developers.binance.com/docs/binance-spot-api-docs/websocket-api/authentication-requests#log-in-with-api-key-signed) ends after sending [`session.logout`](https://developers.binance.com/docs/binance-spot-api-docs/websocket-api/authentication-requests#log-out-of-the-session) method.\n* The subscription is stopped via the [`userDataStream.unsubscribe`](https://developers.binance.com/docs/binance-spot-api-docs/websocket-api/user-data-stream-requests#unsubscribe-from-user-data-stream) method.\n\n**Payload:**\n\n```javascript\n{\n    "subs

In [29]:
def search_docs(query: str) -> str:
    """
    Search the Binance documentation using hybrid search.
    """
    results = hybrid_search(query, num_results=5)

    formatted = []
    for i, r in enumerate(results, 1):
        if isinstance(r, dict):
            text = r.get("text") or r.get("content") or str(r)
        else:
            text = str(r)
        formatted.append(f"Result {i}:\n{text}\n")

    return "\n".join(formatted)

In [30]:
from pydantic_ai import Agent

system_prompt_v1 = """
You are a Binance documentation assistant.

Always use the search_docs tool before answering questions.
Base your answer only on the retrieved documentation.
If the answer is not found, say that clearly.
Do not guess.
"""

agent_v1 = Agent(
    name="binance_agent_v1",
    instructions=system_prompt_v1,
    tools=[search_docs],
    model="gpt-4o-mini"
)

In [31]:
from pydantic_ai import Agent

system_prompt_v2 = """
You are a Binance documentation assistant.

Always use the search_docs tool before answering questions.
Base your answer on the retrieved Binance documentation.
Do not guess when the retrieved results are weak or unclear.

When answering:
- give a direct answer first
- mention important parameters or requirements
- mention authentication/signing requirements if relevant
- cite the filename or section used
""".strip()

agent_v2 = Agent(
    name="binance_agent_v2",
    instructions=system_prompt_v2,
    tools=[search_docs],
    model="gpt-4o-mini"
)

In [32]:
question = "How do I authenticate API requests?"

result_v1 = await agent_v1.run(question)
print("V1:\n", result_v1.output)

result_v2 = await agent_v2.run(question)
print("\nV2:\n", result_v2.output)

V1:
 To authenticate API requests on Binance, you need to follow these key steps:

1. **API Keys**: You need an API key and secret. These can be obtained from your Binance account.

2. **Signature**: For secured API endpoints, you must create a signature using your API secret. This signature is a hashed value created based on the parameters sent in the request, along with a timestamp.

3. **Authenticated Requests**: For authenticated requests, you must include the following parameters in your API call:
   - `apiKey`: Your public API key.
   - `signature`: The generated signature.
   - `timestamp`: The current timestamp (in milliseconds).

4. **Using WebSocket**: If you are using the WebSocket API, you can authenticate your API key after the connection is established. You can do so with the `session.logon` method. After authentication, you can omit the `apiKey` and `signature` for future requests that require them.

5. **Revoke API Key**: If your API key becomes invalid for any reason (

In [102]:
%%writefile logging_utils.py
import json
import secrets
from pathlib import Path
from datetime import datetime
from pydantic_ai.messages import ModelMessagesTypeAdapter

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

def log_entry(agent, messages, source="user", version=None):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "agent_version": version,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source,
    }

def log_interaction_to_file(agent, messages, source="user", version=None):
    entry = log_entry(agent, messages, source, version)

    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath

Overwriting logging_utils.py


In [103]:
from importlib import reload
import logging_utils

reload(logging_utils)
from logging_utils import log_interaction_to_file

print("logging_utils reloaded")

logging_utils reloaded


In [35]:
eval_questions = [
    "How do I authenticate API requests?",
    "How do I place a market order?",
    "What parameters are needed to create an order?",
    "How do I check exchange information?",
    "What are Binance API rate limits?",
    "How do I get account balances?",
    "How do I retrieve recent trades for a symbol?",
    "Which endpoints require signed requests?",
    "What is the difference between limit and market order?",
    "How do I cancel an order?"
]

In [36]:
for q in eval_questions:
    print("\n" + "=" * 80)
    print("QUESTION:", q)

    # Run v1
    result_v1 = await agent_v1.run(q)
    print("\nV1 ANSWER:")
    print(result_v1.output)

    log_file_v1 = log_interaction_to_file(
        agent_v1,
        result_v1.new_messages(),
        source="user",
        version="v1"
    )
    print("Saved V1 log:", log_file_v1)

    # Run v2
    result_v2 = await agent_v2.run(q)
    print("\nV2 ANSWER:")
    print(result_v2.output)

    log_file_v2 = log_interaction_to_file(
        agent_v2,
        result_v2.new_messages(),
        source="user",
        version="v2"
    )
    print("Saved V2 log:", log_file_v2)


QUESTION: How do I authenticate API requests?

V1 ANSWER:
To authenticate API requests on Binance, you can use the following method:

1. **API Key and Signature**: For most requests, you will need to include your API key and a signature that is generated based on your API secret. 

2. **Session Authentication for WebSocket**:
   - If you are using WebSocket, you can authenticate your API key for an active WebSocket session. After authentication, you don't need to include the `apiKey` and `signature` with each request.
   - You authenticate by sending a session logon request:
     ```javascript
     {
         "id": "your-request-id",
         "method": "session.logon",
         "params": {
             "apiKey": "your-api-key",
             "signature": "your-generated-signature",
             "timestamp": your-current-timestamp
         }
     }
     ```
   - After calling `session.logon`, you can omit `apiKey` and `signature` parameters for subsequent requests.

3. **Session Managem

Traceback (most recent call last):
  File "/workspaces/buidling_AI_agents/aihero/project/.venv/lib/python3.12/site-packages/pydantic_ai/models/openai.py", line 740, in _completions_create
    return await self.client.chat.completions.create(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/buidling_AI_agents/aihero/project/.venv/lib/python3.12/site-packages/openai/resources/chat/completions/completions.py", line 2714, in create
    return await self._post(
           ^^^^^^^^^^^^^^^^^
  File "/workspaces/buidling_AI_agents/aihero/project/.venv/lib/python3.12/site-packages/openai/_base_client.py", line 1884, in post
    return await self.request(cast_to, opts, stream=stream, stream_cls=stream_cls)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/workspaces/buidling_AI_agents/aihero/project/.venv/lib/python3.12/site-packages/openai/_base_client.py", line 1669, in request
    raise self._make_status_error_from_response(er

In [38]:
%%writefile evals.py
import json
from pydantic import BaseModel
from pydantic_ai import Agent


class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool


class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>)
to a user question (<QUESTION>).
We also include the full interaction log (<LOG>) for analysis.

For each item, check if the condition is met.

Checklist:
- instructions_follow: The agent followed the system instructions
- answer_relevant: The response directly answers the user's question
- answer_clear: The answer is easy to understand and technically correct
- answer_citations: The answer includes references or source filenames/sections when appropriate
- completeness: The answer covers the key parts of the user's request
- tool_call_search: The search_docs tool was used before answering

Output true/false for each check and provide a short explanation.
""".strip()


eval_agent = Agent(
    name="binance_eval_agent",
    model="gpt-4o-mini",
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist,
)


def load_log_file(log_file):
    with open(log_file, "r", encoding="utf-8") as f_in:
        log_data = json.load(f_in)
    log_data["log_file"] = str(log_file)
    return log_data


def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []

        for original_part in m["parts"]:
            part = original_part.copy()
            kind = part["part_kind"]

            if kind == "user-prompt" and "timestamp" in part:
                del part["timestamp"]

            if kind == "tool-call" and "tool_call_id" in part:
                del part["tool_call_id"]

            if kind == "tool-return":
                if "tool_call_id" in part:
                    del part["tool_call_id"]
                if "metadata" in part:
                    del part["metadata"]
                if "timestamp" in part:
                    del part["timestamp"]
                part["content"] = "RETURN_RESULTS_REDACTED"

            if kind == "text" and "id" in part:
                del part["id"]

            parts.append(part)

        log_simplified.append({
            "kind": m["kind"],
            "parts": parts
        })

    return log_simplified


user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()


async def evaluate_log_record(log_record):
    messages = log_record["messages"]
    instructions = log_record["system_prompt"]
    question = messages[0]["parts"][0]["content"]
    answer = messages[-1]["parts"][0]["content"]

    log_simplified = simplify_log_messages(messages)
    log_json = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log_json,
    )

    result = await eval_agent.run(user_prompt)
    return result.output

Overwriting evals.py


In [39]:
from evals import load_log_file, evaluate_log_record
from pathlib import Path

LOG_DIR = Path("logs")
print("evals ready")

evals ready


In [44]:
log_files = list(LOG_DIR.glob("*.json"))[:5]

eval_results = []

for log_file in log_files:
    log_record = load_log_file(log_file)
    eval_result = await evaluate_log_record(log_record)
    eval_results.append((log_record, eval_result))

print("Done:", len(eval_results))

Done: 5


In [45]:
log_record = load_log_file(log_files[0])
eval_result = await evaluate_log_record(log_record)

print(eval_result.summary)
for check in eval_result.checklist:
    print(check.check_name, "=>", check.check_pass)
    print("reason:", check.justification)
    print()

The agent's response met all the quality checks, providing a complete and accurate answer to the user's question.
instructions_follow => True
reason: The agent followed the system instructions by using the search_docs tool to retrieve documentation before answering the question.

answer_relevant => True
reason: The response directly answers the user's question about the parameters needed to create an order.

answer_clear => True
reason: The answer is clearly structured, easy to understand, and technically correct regarding the parameters.

answer_citations => True
reason: The answer includes a reference to the New Order API documentation link for further details.

completeness => True
reason: The answer includes a comprehensive list of parameters, covering both mandatory and optional parameters, along with their descriptions.

tool_call_search => True
reason: The search_docs tool was used multiple times to find relevant information before providing an answer, as required.



In [109]:
eval_results = []
failed_logs = []

for log_file in list(LOG_DIR.glob("*.json")):
    try:
        log_record = load_log_file(log_file)
        eval_result = await evaluate_log_record(log_record)
        eval_results.append((log_record, eval_result))
        print("Evaluated:", log_file.name)
    except Exception as e:
        failed_logs.append((log_file.name, str(e)))
        print("Failed:", log_file.name, e)

print("Success:", len(eval_results))
print("Failed:", len(failed_logs))

Evaluated: binance_agent_v1_20260328_013415_e5fa55.json
Evaluated: binance_agent_v2_20260328_013447_bc8e7f.json
Evaluated: binance_agent_v2_20260328_013342_d5c32d.json
Evaluated: binance_agent_v2_20260328_013441_4159f5.json
Evaluated: binance_agent_v2_20260328_013552_b47ac4.json
Evaluated: binance_agent_v1_20260328_013249_2e2397.json
Evaluated: binance_agent_v1_20260328_013444_d06064.json
Evaluated: binance_agent_v1_20260328_013310_06c4bb.json
Evaluated: binance_agent_v2_20260328_020037_def460.json
Evaluated: binance_agent_v2_20260328_015913_e6d64c.json
Evaluated: binance_agent_v2_20260328_015849_f4e2d8.json
Evaluated: binance_agent_v2_20260328_015957_21d203.json
Evaluated: binance_agent_v2_20260328_020033_667c72.json
Evaluated: binance_agent_v1_20260328_013525_c50e57.json
Evaluated: binance_agent_v1_20260328_013458_6e6639.json
Evaluated: binance_agent_v2_20260328_013309_443f93.json
Evaluated: binance_agent_v2_20260328_020011_347f4d.json
Evaluated: binance_agent_v2_20260328_015934_cec8

In [110]:
from pathlib import Path
import pandas as pd

rows = []

for log_record, eval_result in eval_results:
    messages = log_record["messages"]

    row = {
        "file": Path(log_record["log_file"]).name,
        "agent_name": log_record.get("agent_name"),
        "agent_version": log_record.get("agent_version"),
        "source": log_record.get("source"),
        "question": messages[0]["parts"][0]["content"],
        "answer": messages[-1]["parts"][0]["content"],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)
    rows.append(row)

df = pd.DataFrame(rows)
df.to_csv("evaluation_results.csv", index=False)

print("saved evaluation_results.csv")
print(df["source"].value_counts())

saved evaluation_results.csv
source
user            12
ai-generated    10
Name: count, dtype: int64


In [111]:
df_user = df[df["source"] == "user"]
df_ai = df[df["source"] == "ai-generated"]

print("User questions:")
print(df_user.mean(numeric_only=True))

print("\nAI-generated questions:")
print(df_ai.mean(numeric_only=True))

User questions:
instructions_follow    1.000000
answer_relevant        1.000000
answer_clear           1.000000
answer_citations       0.583333
completeness           0.750000
tool_call_search       1.000000
dtype: float64

AI-generated questions:
instructions_follow    1.0
answer_relevant        1.0
answer_clear           1.0
answer_citations       1.0
completeness           1.0
tool_call_search       1.0
dtype: float64


In [112]:
%%writefile logging_utils.py
import json
import secrets
from pathlib import Path
from datetime import datetime
from pydantic_ai.messages import ModelMessagesTypeAdapter

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

def log_entry(agent, messages, source="user"):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source,
    }

def log_interaction_to_file(agent, messages, source="user"):
    entry = log_entry(agent, messages, source)

    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath

Overwriting logging_utils.py


In [113]:
from importlib import reload
import logging_utils

reload(logging_utils)
from logging_utils import log_interaction_to_file

print("logging utils reloaded")

logging utils reloaded


In [114]:
from pathlib import Path
list(Path("logs").glob("*.json"))

[PosixPath('logs/binance_agent_v1_20260328_013415_e5fa55.json'),
 PosixPath('logs/binance_agent_v2_20260328_013447_bc8e7f.json'),
 PosixPath('logs/binance_agent_v2_20260328_013342_d5c32d.json'),
 PosixPath('logs/binance_agent_v2_20260328_013441_4159f5.json'),
 PosixPath('logs/binance_agent_v2_20260328_013552_b47ac4.json'),
 PosixPath('logs/binance_agent_v1_20260328_013249_2e2397.json'),
 PosixPath('logs/binance_agent_v1_20260328_013444_d06064.json'),
 PosixPath('logs/binance_agent_v1_20260328_013310_06c4bb.json'),
 PosixPath('logs/binance_agent_v2_20260328_020037_def460.json'),
 PosixPath('logs/binance_agent_v2_20260328_015913_e6d64c.json'),
 PosixPath('logs/binance_agent_v2_20260328_015849_f4e2d8.json'),
 PosixPath('logs/binance_agent_v2_20260328_015957_21d203.json'),
 PosixPath('logs/binance_agent_v2_20260328_020033_667c72.json'),
 PosixPath('logs/binance_agent_v1_20260328_013525_c50e57.json'),
 PosixPath('logs/binance_agent_v1_20260328_013458_6e6639.json'),
 PosixPath('logs/binance_

In [52]:
%%writefile logging_utils.py
import json
import secrets
from pathlib import Path
from datetime import datetime
from pydantic_ai.messages import ModelMessagesTypeAdapter

LOG_DIR = Path("logs")
LOG_DIR.mkdir(exist_ok=True)

def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")

def log_entry(agent, messages, source="user"):
    tools = []
    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source,
    }

def log_interaction_to_file(agent, messages, source="user"):
    entry = log_entry(agent, messages, source)

    ts_str = datetime.now().strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath

Overwriting logging_utils.py


In [53]:
from evals import load_log_file, evaluate_log_record
from pathlib import Path

LOG_DIR = Path("logs")
print("evals ready")

evals ready


In [58]:
log_files = list(LOG_DIR.glob("*.json"))
log_record = load_log_file(log_files[0])

eval_result = await evaluate_log_record(log_record)

print(eval_result.summary)
for check in eval_result.checklist:
    print(check.check_name, "=>", check.check_pass)

The response is well-structured, directly answers the question, follows instructions, and cites appropriate sources.
instructions_follow => True
answer_relevant => True
answer_clear => True
answer_citations => True
completeness => True
tool_call_search => True


In [62]:
import pandas as pd
from pathlib import Path

rows = []

for log_record, eval_result in eval_results:
    messages = log_record["messages"]

    row = {
        "file": Path(log_record["log_file"]).name,
        "agent_name": log_record.get("agent_name"),
        "agent_version": log_record.get("agent_version"),
        "source": log_record.get("source"),
        "question": messages[0]["parts"][0]["content"],
        "answer": messages[-1]["parts"][0]["content"],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)
    rows.append(row)

df_evals = pd.DataFrame(rows)
df_evals.to_csv("evaluation_results.csv", index=False)
print("saved evaluation_results.csv")

saved evaluation_results.csv


In [65]:
df = pd.DataFrame(rows)
df.to_csv("evaluation_results.csv", index=False)

In [66]:
df.mean(numeric_only=True)

instructions_follow    1.000000
answer_relevant        0.916667
answer_clear           1.000000
answer_citations       0.583333
completeness           0.833333
tool_call_search       1.000000
dtype: float64

In [67]:
from importlib import reload
import logging_utils
import evals

reload(logging_utils)
reload(evals)

from logging_utils import log_interaction_to_file
from evals import load_log_file, evaluate_log_record

print("modules reloaded")

modules reloaded


In [68]:
df_v1 = df_evals[df_evals["agent_version"] == "v1"]
df_v2 = df_evals[df_evals["agent_version"] == "v2"]

print("V1 rows:", len(df_v1))
print("V2 rows:", len(df_v2))

print("\nV1 averages:")
print(df_v1.mean(numeric_only=True))

print("\nV2 averages:")
print(df_v2.mean(numeric_only=True))

V1 rows: 6
V2 rows: 6

V1 averages:
instructions_follow    1.000000
answer_relevant        0.833333
answer_clear           1.000000
answer_citations       0.333333
completeness           0.833333
tool_call_search       1.000000
dtype: float64

V2 averages:
instructions_follow    1.000000
answer_relevant        1.000000
answer_clear           1.000000
answer_citations       0.833333
completeness           0.833333
tool_call_search       1.000000
dtype: float64


In [69]:
failure_cols = [
    "instructions_follow",
    "answer_relevant",
    "answer_clear",
    "answer_citations",
    "completeness",
    "tool_call_search"
]

failed_rows = df_evals[
    ~(df_evals[failure_cols].all(axis=1))
]

failed_rows[["file", "question"] + failure_cols]

,file,question,instructions_follow,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
1,binance_agent_v2_20260328_013447_bc8e7f.json,How do I check exchange information?,True,True,True,False,False,True
5,binance_agent_v1_20260328_013249_2e2397.json,How do I authenticate API requests?,True,True,True,False,True,True
6,binance_agent_v1_20260328_013444_d06064.json,How do I check exchange information?,True,False,True,False,False,True
7,binance_agent_v1_20260328_013310_06c4bb.json,How do I place a market order?,True,True,True,False,True,True
8,binance_agent_v1_20260328_013525_c50e57.json,How do I get account balances?,True,True,True,False,True,True


In [70]:
df_evals[df_evals["answer_citations"] == False][["file", "question", "answer"]]

,file,question,answer
1,binance_agent_v2_20260328_013447_bc8e7f.json,How do I check exchange information?,"To check exchange information on Binance, you ..."
5,binance_agent_v1_20260328_013249_2e2397.json,How do I authenticate API requests?,"To authenticate API requests on Binance, you c..."
6,binance_agent_v1_20260328_013444_d06064.json,How do I check exchange information?,The search did not yield specific instructions...
7,binance_agent_v1_20260328_013310_06c4bb.json,How do I place a market order?,The search did not return any relevant informa...
8,binance_agent_v1_20260328_013525_c50e57.json,How do I get account balances?,"To get your account balances on Binance, you c..."


In [71]:
df_evals[df_evals["completeness"] == False][["file", "question", "answer"]]

,file,question,answer
1,binance_agent_v2_20260328_013447_bc8e7f.json,How do I check exchange information?,"To check exchange information on Binance, you ..."
6,binance_agent_v1_20260328_013444_d06064.json,How do I check exchange information?,The search did not yield specific instructions...


In [72]:
df_v1.to_csv("evaluation_results_v1.csv", index=False)
df_v2.to_csv("evaluation_results_v2.csv", index=False)

In [73]:
print("V1")
print(df_v1.mean(numeric_only=True))

print("V2")
print(df_v2.mean(numeric_only=True))

V1
instructions_follow    1.000000
answer_relevant        0.833333
answer_clear           1.000000
answer_citations       0.333333
completeness           0.833333
tool_call_search       1.000000
dtype: float64
V2
instructions_follow    1.000000
answer_relevant        1.000000
answer_clear           1.000000
answer_citations       0.833333
completeness           0.833333
tool_call_search       1.000000
dtype: float64


In [74]:
print("\nV1 length:", len(result_v1.output))
print("V2 length:", len(result_v2.output))


V1 length: 1124
V2 length: 1652


In [75]:
from pathlib import Path

Path("evaluation_results.csv").exists()

True

In [76]:
import pandas as pd

df = pd.read_csv("evaluation_results.csv")
df.head()

,file,agent_name,agent_version,source,question,answer,instructions_follow,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
0,binance_agent_v1_20260328_013415_e5fa55.json,binance_agent_v1,v1,user,What parameters are needed to create an order?,"To create an order on Binance, you need the fo...",True,True,True,True,True,True
1,binance_agent_v2_20260328_013447_bc8e7f.json,binance_agent_v2,v2,user,How do I check exchange information?,"To check exchange information on Binance, you ...",True,True,True,False,False,True
2,binance_agent_v2_20260328_013342_d5c32d.json,binance_agent_v2,v2,user,How do I place a market order?,To place a market order on Binance via the API...,True,True,True,True,True,True
3,binance_agent_v2_20260328_013441_4159f5.json,binance_agent_v2,v2,user,What parameters are needed to create an order?,"To create an order on Binance, the following p...",True,True,True,True,True,True
4,binance_agent_v2_20260328_013552_b47ac4.json,binance_agent_v2,v2,user,How do I get account balances?,"To get your account balances on Binance, you c...",True,True,True,True,True,True


In [77]:
df.shape

(12, 12)

In [78]:
df_v1 = df[df["agent_version"] == "v1"]
df_v2 = df[df["agent_version"] == "v2"]

print("V1:")
print(df_v1.mean(numeric_only=True))

print("\nV2:")
print(df_v2.mean(numeric_only=True))

V1:
instructions_follow    1.000000
answer_relevant        0.833333
answer_clear           1.000000
answer_citations       0.333333
completeness           0.833333
tool_call_search       1.000000
dtype: float64

V2:
instructions_follow    1.000000
answer_relevant        1.000000
answer_clear           1.000000
answer_citations       0.833333
completeness           0.833333
tool_call_search       1.000000
dtype: float64


In [79]:
df[df["completeness"] == False]

,file,agent_name,agent_version,source,question,answer,instructions_follow,answer_relevant,answer_clear,answer_citations,completeness,tool_call_search
1,binance_agent_v2_20260328_013447_bc8e7f.json,binance_agent_v2,v2,user,How do I check exchange information?,"To check exchange information on Binance, you ...",True,True,True,False,False,True
6,binance_agent_v1_20260328_013444_d06064.json,binance_agent_v1,v1,user,How do I check exchange information?,The search did not yield specific instructions...,True,False,True,False,False,True


In [101]:
df_user = df[df["source"] == "user"]
df_ai = df[df["source"] == "ai-generated"]

print("User questions:")
print(df_user.mean(numeric_only=True))

print("\nAI-generated questions:")
print(df_ai.mean(numeric_only=True))

User questions:
instructions_follow       1.000000
answer_relevant           0.916667
answer_clear              1.000000
answer_citations          0.583333
completeness              0.833333
tool_call_search          1.000000
answer_length          1359.333333
dtype: float64

AI-generated questions:
instructions_follow   NaN
answer_relevant       NaN
answer_clear          NaN
answer_citations      NaN
completeness          NaN
tool_call_search      NaN
answer_length         NaN
dtype: float64


In [81]:
df.mean(numeric_only=True)
#spot fake scores

instructions_follow    1.000000
answer_relevant        0.916667
answer_clear           1.000000
answer_citations       0.583333
completeness           0.833333
tool_call_search       1.000000
dtype: float64

In [82]:
import json

with open(list(Path("logs").glob("*.json"))[0]) as f:
    log = json.load(f)

log

{'agent_name': 'binance_agent_v1',
 'agent_version': 'v1',
 'system_prompt': ['\nYou are a Binance documentation assistant.\n\nAlways use the search_docs tool before answering questions.\nBase your answer only on the retrieved documentation.\nIf the answer is not found, say that clearly.\nDo not guess.\n'],
 'provider': 'openai',
 'model': 'gpt-4o-mini',
 'tools': ['search_docs'],
 'messages': [{'parts': [{'content': 'What parameters are needed to create an order?',
     'timestamp': '2026-03-28T01:33:42.380963+00:00',
     'part_kind': 'user-prompt'}],
   'timestamp': '2026-03-28T01:33:42.381137+00:00',
   'instructions': 'You are a Binance documentation assistant.\n\nAlways use the search_docs tool before answering questions.\nBase your answer only on the retrieved documentation.\nIf the answer is not found, say that clearly.\nDo not guess.',
   'kind': 'request',
   'run_id': '7f82b921-1fac-433e-b48d-53436e6d3d43',
   'metadata': None},
  {'parts': [{'tool_name': 'search_docs',
    

In [83]:
import json

with open(list(Path("logs").glob("*.json"))[0]) as f:
    log = json.load(f)

log

{'agent_name': 'binance_agent_v1',
 'agent_version': 'v1',
 'system_prompt': ['\nYou are a Binance documentation assistant.\n\nAlways use the search_docs tool before answering questions.\nBase your answer only on the retrieved documentation.\nIf the answer is not found, say that clearly.\nDo not guess.\n'],
 'provider': 'openai',
 'model': 'gpt-4o-mini',
 'tools': ['search_docs'],
 'messages': [{'parts': [{'content': 'What parameters are needed to create an order?',
     'timestamp': '2026-03-28T01:33:42.380963+00:00',
     'part_kind': 'user-prompt'}],
   'timestamp': '2026-03-28T01:33:42.381137+00:00',
   'instructions': 'You are a Binance documentation assistant.\n\nAlways use the search_docs tool before answering questions.\nBase your answer only on the retrieved documentation.\nIf the answer is not found, say that clearly.\nDo not guess.',
   'kind': 'request',
   'run_id': '7f82b921-1fac-433e-b48d-53436e6d3d43',
   'metadata': None},
  {'parts': [{'tool_name': 'search_docs',
    

In [84]:
df["answer_length"] = df["answer"].apply(len)

df_v1 = df[df["agent_version"] == "v1"]
df_v2 = df[df["agent_version"] == "v2"]

print(df_v1["answer_length"].mean())
print(df_v2["answer_length"].mean())

1139.6666666666667
1579.0


In [85]:
df["source"].value_counts(dropna=False)

source
user    12
Name: count, dtype: int64

In [86]:
len(df[df["source"] == "ai-generated"])

0

In [87]:
import json
import random
from pydantic import BaseModel
from pydantic_ai import Agent

In [88]:
class QuestionsList(BaseModel):
    questions: list[str]

In [89]:
question_generation_prompt = """
You are helping create test questions for an AI agent that answers questions about Binance API documentation.

Based on the provided documentation snippets, generate realistic questions that a user might ask.

The questions should:
- be natural and varied
- range from simple to complex
- include authentication, orders, rate limits, account info, market data, and signing
- generate one question per input record
""".strip()

In [90]:
question_generator = Agent(
    name="binance_question_generator",
    instructions=question_generation_prompt,
    model="gpt-4o-mini",
    output_type=QuestionsList,
)

In [91]:
import json
import random

with open("my_chunks_sections.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

sample = random.sample(chunks, 10)

In [92]:
sample[0]

{'filename': 'binance-spot-api-docs-master/testnet/web-socket-api.md',
 'section': '## User Data Stream requests\n\n### User Data Stream subscription\n\n<a id=general_info_user_data_stream_subscriptions></a>\n**General information:**\n\n* [User Data Stream](user-data-stream.md) subscriptions allow you to receive all the events related to a given account on a WebSocket connection.\n* There are 2 ways to start a subscription:\n  * If you have an authenticated session, then you can subscribe to events for that authenticated account using [`userDataStream.subscribe`](#user-data-stream-subscribe).\n  * In any session, authenticated or not, you can subscribe to events for one or more accounts for which you can provide an API Key signature, using [`userDataStream.subscribe.signature`](#user-data-signature).\n  * You can have only one active subscription for a given account on a given connection.\n* Subscriptions are identified by a `subscriptionId` which is returned when starting the subscrip

In [93]:
def clean_section(text):
    return text.replace("\n", " ").strip()

prompt_docs = [clean_section(d["section"]) for d in sample]

In [94]:
prompt = json.dumps(prompt_docs, ensure_ascii=False)
result = await question_generator.run(prompt)
ai_questions = result.output.questions
ai_questions

['How can I authenticate my session when using the Binance WebSocket API?',
 'What is the maximum number of active User Data Stream subscriptions I can have on a single session?',
 'What should I do if I receive an error while trying to subscribe to the User Data Stream?',
 'How do I subscribe to the User Data Stream using the signature method?',
 'What parameters do I need to consider for signing requests to the Binance API?',
 'What happens if I exceed the rate limit for order submissions on Binance?',
 'How can I check how many active subscriptions I have in my session with the Binance WebSocket API?',
 'What is the response format for the User Data Stream events?',
 'How do the rate limits work for the Binance API?',
 'Can I receive market data without an authenticated session on Binance?']

In [95]:
for q in ai_questions:
    print(q)

How can I authenticate my session when using the Binance WebSocket API?
What is the maximum number of active User Data Stream subscriptions I can have on a single session?
What should I do if I receive an error while trying to subscribe to the User Data Stream?
How do I subscribe to the User Data Stream using the signature method?
What parameters do I need to consider for signing requests to the Binance API?
What happens if I exceed the rate limit for order submissions on Binance?
How can I check how many active subscriptions I have in my session with the Binance WebSocket API?
What is the response format for the User Data Stream events?
How do the rate limits work for the Binance API?
Can I receive market data without an authenticated session on Binance?


In [104]:
for q in ai_questions:
    print("\nAI QUESTION:", q)

    result = await agent_v2.run(q)

    log_file = log_interaction_to_file(
        agent_v2,
        result.new_messages(),
        source="ai-generated",
        version="v2"
    )

    print("Saved:", log_file)


AI QUESTION: How can I authenticate my session when using the Binance WebSocket API?
Saved: logs/binance_agent_v2_20260328_015849_f4e2d8.json

AI QUESTION: What is the maximum number of active User Data Stream subscriptions I can have on a single session?
Saved: logs/binance_agent_v2_20260328_015853_dddd03.json

AI QUESTION: What should I do if I receive an error while trying to subscribe to the User Data Stream?
Saved: logs/binance_agent_v2_20260328_015901_689049.json

AI QUESTION: How do I subscribe to the User Data Stream using the signature method?
Saved: logs/binance_agent_v2_20260328_015913_e6d64c.json

AI QUESTION: What parameters do I need to consider for signing requests to the Binance API?
Saved: logs/binance_agent_v2_20260328_015934_cec8e1.json

AI QUESTION: What happens if I exceed the rate limit for order submissions on Binance?
Saved: logs/binance_agent_v2_20260328_015948_348301.json

AI QUESTION: How can I check how many active subscriptions I have in my session with th

In [106]:
import json
from pathlib import Path
from collections import Counter

sources = []
for f in Path("logs").glob("*.json"):
    with open(f, "r", encoding="utf-8") as infile:
        sources.append(json.load(infile).get("source"))

Counter(sources)

Counter({'user': 12, 'ai-generated': 10})

In [107]:
df["source"].value_counts()

source
user    12
Name: count, dtype: int64

In [118]:
df_ai = df[df["source"] == "ai-generated"]
df_ai.mean(numeric_only=True)

instructions_follow    1.0
answer_relevant        1.0
answer_clear           1.0
answer_citations       1.0
completeness           1.0
tool_call_search       1.0
dtype: float64

In [117]:
# user questions
df_user = df[df["source"] == "user"]

# AI-generated questions
df_ai = df[df["source"] == "ai-generated"]

# split by version
df_v1 = df[df["agent_version"] == "v1"]
df_v2 = df[df["agent_version"] == "v2"]

In [119]:
print("V1 performance:")
print(df_v1.mean(numeric_only=True))

print("\nV2 performance:")
print(df_v2.mean(numeric_only=True))

V1 performance:
instructions_follow    1.000000
answer_relevant        1.000000
answer_clear           1.000000
answer_citations       0.333333
completeness           0.666667
tool_call_search       1.000000
dtype: float64

V2 performance:
instructions_follow    1.0000
answer_relevant        1.0000
answer_clear           1.0000
answer_citations       0.9375
completeness           0.9375
tool_call_search       1.0000
dtype: float64


In [120]:
print("User questions:")
print(df_user.mean(numeric_only=True))

print("\nAI-generated questions:")
print(df_ai.mean(numeric_only=True))

User questions:
instructions_follow    1.000000
answer_relevant        1.000000
answer_clear           1.000000
answer_citations       0.583333
completeness           0.750000
tool_call_search       1.000000
dtype: float64

AI-generated questions:
instructions_follow    1.0
answer_relevant        1.0
answer_clear           1.0
answer_citations       1.0
completeness           1.0
tool_call_search       1.0
dtype: float64
